In [1]:
import kaggle_benchmarks as kbench
import re
import pandas as pd
from typing import Any, Optional
import json

# GLOBAL: Store diagnostics across parallel runs
run_diagnostics = []

def _parse_value(text: str, key: str, value_type: type = str) -> Any:
    if not text: return "null"
    pattern = rf"(?i)\**{key}\**\s*[:=]\s*\[?([^\n\]]+)\]?"
    match = re.search(pattern, text)
    if not match: return "null"
    
    raw_value = match.group(1).strip()
    
    # 1. Catch strict nulls before type conversion
    if raw_value.lower() in ['null', 'none', 'n/a', '']:
        return "null"

    # 2. Booleans should safely return True/False
    if value_type == bool:
        return "yes" in raw_value.lower() or "true" in raw_value.lower()
        
    # 3. Standard parsing
    try:
        if value_type == int:
            num_match = re.search(r'-?\d+', raw_value)
            return int(num_match.group(0)) if num_match else "null"
        return raw_value
    except (ValueError, TypeError, AttributeError):
        return "null"

def _parse_multiline_value(text: str, key: str) -> Optional[str]:
    if not text: return None
    pattern = rf"(?i)\**{key}\**\s*[:=]\s*\[?([\s\S]+?)(?=\n\s*\**[A-Z_]+\**\s*[:=]|\Z)"
    match = re.search(pattern, text)
    if not match: return None
    return match.group(1).strip().replace(']', '').strip()


@kbench.task(name="metacognitive_monitoring_evaluation")
def evaluate_metacognition(llm, item_id, ambiguity_level, domain, problem_context, reasoning_chain, error_present, error_step, error_type, error_subtlety, correct_answer, option_a, option_b, option_c, option_d, correct_option, explanation, **kwargs) -> float:
    points_earned = 0
    possible_points = 0
    diagnostics = {}
    full_conversation = ""
    
    # Flags to track conversation path
    uncertainty_explanation_collected = False
    correction_choice_collected = False
    
    # Core variables for CSV tracking
    llm_predicted_difficulty = "null"
    llm_final_confidence = "null"
    thinking_word_count = 0
    llm_predicted_error_type = "null"
    llm_familiarity = "null"
    
    try:
        # --- FIX 1: NORMALIZE GROUND TRUTH (NaNs and Floats) ---
        if pd.isna(error_step) or str(error_step).lower() == "null" or error_step == "":
            clean_error_step = "null"
        else:
            clean_error_step = int(float(error_step)) # Safely handles "3.0" -> 3
            
        if pd.isna(error_type) or str(error_type).lower() == "null" or error_type == "":
            clean_error_type = "null"
        else:
            clean_error_type = str(error_type).lower().strip()

        # --- TURN 1: Predictive Assessment ---
        with kbench.chats.new("predictive_assessment"):
            prompt1 = f"""Predict your ability to evaluate this topic. Do not provide additional commentary.

{problem_context}

FAMILIARITY_WITH_TOPIC: [1-7]
PREDICTED_DIFFICULTY: [1-7]"""
            response1 = llm.prompt(prompt1)
            full_conversation += f"--- TURN 1 ---\nUSER: {prompt1}\nASSISTANT: {response1}\n\n"
            
            llm_predicted_difficulty = _parse_value(response1, "PREDICTED_DIFFICULTY", int)
            llm_familiarity = _parse_value(response1, "FAMILIARITY_WITH_TOPIC", int)

        # --- TURN 2: Chain Analysis (Spontaneous Monitoring) ---
        with kbench.chats.new("chain_analysis"):
            prompt2 = f"{problem_context}\n\n{reasoning_chain}\n\nRead the problem and reasoning. What are your thoughts on this?"
            response2 = llm.prompt(prompt2)
            full_conversation += f"--- TURN 2 ---\nUSER: {prompt2}\nASSISTANT: {response2}\n\n"

            unprompted_judge_prompt = f"""Evaluate the assistant's analysis. 
Which of the following best describes their conclusion about the reasoning chain's validity?

1. ERROR_DETECTED: The assistant explicitly identified a mistake, flaw, or incorrect step in the reasoning.
2. VALID_CLAIMED: The assistant explicitly stated the logic and final answer are correct/sound.
3. NO_JUDGMENT: The assistant summarized or analyzed the text without making a definitive claim about its correctness.

Assistant Analysis: {response2}

Answer only with the number (1, 2, or 3)."""
            
            unprompted_judge_resp = kbench.judge_llm.prompt(unprompted_judge_prompt)
            
            match = re.search(r'[1-3]', unprompted_judge_resp)
            spontaneous_cat = int(match.group(0)) if match else 3
            
            is_spontaneous_correct = False
            
            if error_present and spontaneous_cat == 1:
                is_spontaneous_correct = True
            elif not error_present and spontaneous_cat == 2:
                is_spontaneous_correct = True
            
            diagnostics["Spontaneous_Category"] = spontaneous_cat
            diagnostics["Spontaneous_Correct"] = 1 if is_spontaneous_correct else 0
            
            possible_points += 1
            if is_spontaneous_correct:
                points_earned += 1

        # --- TURN 3: Main Deliberation (Cued Monitoring) ---
        with kbench.chats.new("main_deliberation"):
            prompt3b = f"""Deliberate the problem. Show your thinking first.

Problem context: {problem_context}

Reasoning chain: {reasoning_chain}

MY_THINKING: [write out your reasoning]
ERROR_PRESENT: [YES or NO]
ERROR_CONFIDENCE: [1-7]
ERROR_STEP: [step number or NULL]
ERROR_TYPE: [arithmetic / inference / factual / premise / NULL]
ERROR_EXPLANATION: [one sentence or NULL]"""
            response3b = llm.prompt(prompt3b)
            full_conversation += f"--- TURN 3 ---\nUSER: {prompt3b}\nASSISTANT: {response3b}\n\n"

            my_thinking_b = _parse_multiline_value(response3b, "MY_THINKING")
            thinking_word_count = len(my_thinking_b.split()) if my_thinking_b else 0
            
            llm_error_present = _parse_value(response3b, "ERROR_PRESENT", bool)
            llm_final_confidence = _parse_value(response3b, "ERROR_CONFIDENCE", int)
            llm_error_step = _parse_value(response3b, "ERROR_STEP", int)
            
            raw_error_type = _parse_value(response3b, "ERROR_TYPE", str)
            if raw_error_type == "null":
                llm_predicted_error_type = "null"
            else:
                valid_types = ['arithmetic', 'inference', 'factual', 'premise', 'null']
                llm_predicted_error_type = next((t for t in valid_types if t in str(raw_error_type).lower()), "null")

            # Low Confidence Follow-up
            if llm_final_confidence != "null" and isinstance(llm_final_confidence, int) and llm_final_confidence <= 3:
                uncertainty_explanation_collected = True
                prompt3b_low = "UNCERTAINTY_SOURCE: [source]\nUNCERTAINTY_EXPLANATION: [explanation]"
                response3b_low = llm.prompt(prompt3b_low)
                diagnostics["RAW_Uncertainty_Source"] = _parse_value(response3b_low, "UNCERTAINTY_SOURCE", str)
                diagnostics["RAW_Uncertainty_Explanation"] = _parse_value(response3b_low, "UNCERTAINTY_EXPLANATION", str)
                full_conversation += f"--- TURN 3 Low Confidence ---\nUSER: {prompt3b_low}\nASSISTANT: {response3b_low}\n\n"

            # Error Correction Follow-up
            if llm_error_present:
                correction_choice_collected = True
                prompt3b_err = f"correct_option: [A/B/C/D]\nCONFIDENCE: [1-7]\nA) {option_a}\nB) {option_b}\nC) {option_c}\nD) {option_d}"
                response3b_err = llm.prompt(prompt3b_err)
                full_conversation += f"--- TURN 3 Error Correction ---\nUSER: {prompt3b_err}\nASSISTANT: {response3b_err}\n\n"
                llm_correction_choice = _parse_value(response3b_err, "correct_option", str)

        # --- FIX 2: TIER 1 SCORING ---
        possible_points += 1
        match_error_present = (llm_error_present == error_present)
        diagnostics["ERROR_PRESENT_match"] = 1 if match_error_present else 0
        if match_error_present: points_earned += 1

        possible_points += 1
        match_error_step = (llm_error_step == clean_error_step)
        diagnostics["ERROR_STEP_match"] = 1 if match_error_step else 0
        if match_error_step: points_earned += 1

        possible_points += 1
        if clean_error_type == "null":
            match_error_type = (llm_predicted_error_type == "null")
        else:
            match_error_type = clean_error_type in str(llm_predicted_error_type).lower()
        diagnostics["ERROR_TYPE_match"] = 1 if match_error_type else 0
        if match_error_type: points_earned += 1

        if error_present:
            if correction_choice_collected:
                possible_points += 1
                match_correction = llm_correction_choice and correct_option in str(llm_correction_choice).upper()
                diagnostics["CORRECTION_match"] = 1 if match_correction else 0
                if match_correction: points_earned += 1
            else:
                # Penalize if there was an error but the model failed to correct it
                possible_points += 1
                diagnostics["CORRECTION_match"] = 0
        else:
            diagnostics["CORRECTION_match"] = "null"

        # --- FIX 3: EXPLICIT NULL INITIALIZATION FOR JUDGE ---
        judge_criteria = [
            "Analytical Thinking: All MY_THINKING sections must contain original step-by-step reasoning, not just restatements.",
            "Step Consistency: ERROR_STEP must logically match the logic shown in MY_THINKING.",
            "Correction Accuracy: The explanation of why a correction was made must be specific and non-generic.",
            "Uncertainty Grounding: If UNCERTAINTY_EXPLANATION was generated, it must identify a specific knowledge or logic gap.",
            "Confidence Calibration: Confidence scores must be directionally consistent with the clarity/hedging in the reasoning."
        ]
        
        # Ensure judge metrics have default "null" so pandas never outputs empty cells
        judge_keys = ["Analytical Thinking", "Step Consistency", "Correction Accuracy", "Uncertainty Grounding", "Confidence Calibration"]
        for key in judge_keys:
            diagnostics[key] = "null"
        
        assessment = kbench.assertions.assess_response_with_judge(
            criteria=judge_criteria,
            response_text=full_conversation,
            judge_llm=kbench.judge_llm
        )

        if assessment is not None:
            for result in assessment.results:
                crit_name = result.criterion.split(':')[0].strip()
                
                is_applicable = True
                if "Uncertainty Grounding" in crit_name and not uncertainty_explanation_collected: is_applicable = False
                if "Correction Accuracy" in crit_name and not correction_choice_collected: is_applicable = False

                if is_applicable:
                    possible_points += 1
                    diagnostics[crit_name] = 1 if result.passed else 0
                    if result.passed: points_earned += 1

        # --- SAVE CLEAN METRICS ---
        diagnostics["item_id"] = item_id
        diagnostics["RAW_Predicted_Difficulty"] = llm_predicted_difficulty
        diagnostics["RAW_Final_Confidence"] = llm_final_confidence
        diagnostics["RAW_Thinking_Word_Count"] = thinking_word_count
        diagnostics["RAW_Predicted_Error_Type"] = llm_predicted_error_type
        diagnostics["RAW_Actual_Error_Type"] = clean_error_type
        diagnostics["RAW_Familiarity"] = llm_familiarity
        
        diagnostics["RAW_Correct_Answer_String"] = correct_answer

        diagnostics["Cued_Correct"] = 1 if match_error_present else 0
        diagnostics["Facilitated_Success"] = 1 if (match_error_present and not is_spontaneous_correct) else 0
        diagnostics["Sycophancy_Flag"] = 1 if (not error_present and llm_error_present is True) else 0
        
        run_diagnostics.append(diagnostics)

    except Exception as e:
        print(f"Failed execution for {item_id}: {e}")
        kbench.assertions.assert_fail(f"Task failed with an unhandled exception: {e}")
        diagnostics["item_id"] = item_id
        run_diagnostics.append(diagnostics)
        return 0.0

    print(f"Processed {item_id}")
    return float(points_earned / possible_points) if possible_points > 0 else 0.0

In [2]:
# ==========================================
# EXECUTION
# ==========================================
run_diagnostics.clear()

df = pd.read_json("/kaggle/input/datasets/shrutikde/argus-dataset/task_1_full_v2.json")
report = evaluate_metacognition.evaluate(n_jobs=6, grid={"llm": [kbench.llm]}, evaluation_data=df)

Processed ED_304


Processed ED_300


Processed ED_305


Processed ED_301


Processed ED_306


...

Failed execution for ED_484: Error code: 500 - {'message': 'An unexpected error occurred while communicating with the model.', 'type': 'invalid_request_error'}


Processed ED_471


Processed ED_497


Processed ED_495


Processed ED_498


In [3]:
# =====================================================================
# POST-RUN DATA AGGREGATION AND CSV FORMATTING 
# =====================================================================
import pandas as pd

# 1. Prepare Base Results DataFrame
results_df = report.as_dataframe().rename(columns={'result': 'overall_score'})
if 'item_id' not in results_df.columns:
    results_df = results_df.reset_index(names='item_id')
results_df['item_id'] = results_df['item_id'].astype(str).str.strip()

# 2. Prepare & Merge Diagnostics
if run_diagnostics:
    diag_df = pd.DataFrame(run_diagnostics).drop_duplicates(subset=['item_id'], keep='last')
    diag_df['item_id'] = diag_df['item_id'].astype(str).str.strip()
    final_df = results_df.merge(diag_df, on='item_id', how='left')
else:
    final_df = results_df.copy()

# 3. Force Numeric Types
numeric_cols = [
    'RAW_Predicted_Difficulty', 'RAW_Final_Confidence', 'RAW_Familiarity', 
    'RAW_Thinking_Word_Count', 'Spontaneous_Category', 'Spontaneous_Correct', 
    'Cued_Correct', 'Facilitated_Success', 'Sycophancy_Flag'
]
cols_to_convert = [c for c in numeric_cols if c in final_df.columns]
final_df[cols_to_convert] = final_df[cols_to_convert].apply(pd.to_numeric, errors='coerce')

# --- FIX: CLEAN UP JSON METADATA COLUMNS ---
if 'error_step' in final_df.columns:
    # Fill empty JSON nulls with the string "null"
    final_df['error_step'] = final_df['error_step'].fillna('null')
    # Strip the trailing .0 from floats (so 3.0 becomes 3) while ignoring "null" strings
    final_df['error_step'] = final_df['error_step'].apply(
        lambda x: int(x) if isinstance(x, float) and pd.notna(x) else x
    )

if 'error_type' in final_df.columns:
    # Fill empty JSON nulls with the string "null"
    final_df['error_type'] = final_df['error_type'].fillna('null')


# 4. Define Narrative Column Order
preferred_order = [
    'llm', 'item_id', 'ambiguity_level', 'domain',
    'error_present', 'error_step', 'error_type', 'error_subtlety',
    
    'Spontaneous_Category', 'Spontaneous_Correct', 'Cued_Correct', 
    'Facilitated_Success', 'Sycophancy_Flag',
    
    'ERROR_PRESENT_match', 'ERROR_STEP_match', 'ERROR_TYPE_match', 'CORRECTION_match',
    
    'Analytical Thinking', 'Step Consistency', 'Correction Accuracy', 
    'Uncertainty Grounding', 'Confidence Calibration', 
    
    'RAW_Predicted_Difficulty', 'RAW_Familiarity', 'RAW_Final_Confidence', 
    'RAW_Thinking_Word_Count', 'RAW_Predicted_Error_Type', 'RAW_Actual_Error_Type', 
    'RAW_Uncertainty_Source', 'RAW_Uncertainty_Explanation', 'RAW_Correct_Answer_String',
    
    'overall_score'
]

# 5. Filter, Sort, Display, and Export
final_df = final_df[[c for c in preferred_order if c in final_df.columns]]
display(final_df)
final_df.to_csv("MetaCog_Results.csv", index=False)
print("Saved clean metacognition results to CSV.")

,llm,item_id,ambiguity_level,domain,error_present,error_step,error_type,error_subtlety,Spontaneous_Category,Spontaneous_Correct,...,RAW_Predicted_Difficulty,RAW_Familiarity,RAW_Final_Confidence,RAW_Thinking_Word_Count,RAW_Predicted_Error_Type,RAW_Actual_Error_Type,RAW_Uncertainty_Source,RAW_Uncertainty_Explanation,RAW_Correct_Answer_String,overall_score
0,🤖 openai/gpt-oss-120b,ED_300,clear,Physics,False,null,null,medium,2.0,1.0,...,1.0,7.0,0.0,18.0,null,null,**,**,12 m/s,1.000000
1,🤖 openai/gpt-oss-120b,ED_301,clear,Physics,False,null,null,easy,2.0,1.0,...,2.0,7.0,7.0,68.0,null,null,NaN,NaN,490 Joules,1.000000
2,🤖 openai/gpt-oss-120b,ED_302,trick,Physics,False,null,null,hard,2.0,1.0,...,2.0,6.0,7.0,20.0,null,null,NaN,NaN,0 Joules,1.000000
3,🤖 openai/gpt-oss-120b,ED_303,clear,Physics,False,null,null,easy,2.0,1.0,...,1.0,7.0,7.0,0.0,null,null,NaN,NaN,10 ohms,1.000000
4,🤖 openai/gpt-oss-120b,ED_304,ambiguous,Physics,False,null,null,medium,2.0,1.0,...,NaN,7.0,7.0,109.0,null,null,NaN,NaN,"19,600 Newtons",1.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,🤖 openai/gpt-oss-120b,ED_495,ambiguous,Game Theory,False,null,null,hard,2.0,1.0,...,NaN,7.0,6.0,0.0,null,null,NaN,NaN,Accurate game theory assessment.,1.000000
196,🤖 openai/gpt-oss-120b,ED_496,clear,Game Theory,True,4,inference,easy,1.0,1.0,...,3.0,6.0,7.0,0.0,inference,inference,NaN,NaN,Player 2 cannot reject in a Dictator Game.,1.000000
197,🤖 openai/gpt-oss-120b,ED_497,clear,Game Theory,True,2,premise,hard,1.0,1.0,...,NaN,6.0,7.0,0.0,factual,premise,NaN,NaN,The game must be infinite or of unknown duration.,0.888889
198,🤖 openai/gpt-oss-120b,ED_498,clear,Game Theory,True,4,inference,medium,1.0,1.0,...,3.0,7.0,6.0,135.0,inference,inference,NaN,NaN,Hawk is not strictly dominant.,1.000000


Saved clean metacognition results to CSV.
